In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

FIGURES = Path('../outputs/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

GOVT_COLOR = '#4c72b0'
PVT_COLOR  = '#dd8452'
PTR_CAP    = 100   # 104 schools exceed this; max raw value is 1,196 (data errors)

df = pd.read_csv('../Data:/school_resources.csv')

# Cap PTR: keep flag for reporting, create clean column
n_extreme = (df['pupil_teacher_ratio'] > PTR_CAP).sum()
df['ptr_clean'] = df['pupil_teacher_ratio'].where(df['pupil_teacher_ratio'] <= PTR_CAP)

print(f'Schools: {len(df):,}  |  Government: {(df.school_type=="Government").sum():,}  '
      f'|  Private: {(df.school_type=="Private").sum():,}')
print(f'PTR values > {PTR_CAP} excluded from PTR charts: {n_extreme} schools')

## Figure 1 — Key Operational Metrics: Government vs Private Schools

Three school-level metrics directly shape the instructional environment a child experiences. Taken together, they provide a resource-side explanation for the learning outcome gaps documented in notebooks 01–05:

- **Pupil–teacher ratio (PTR)**: a higher PTR means each teacher is responsible for more students, reducing individualised attention. Government school PTRs are nearly twice the private school average, and the distribution is highly right-skewed (some schools report ratios above 100, almost certainly reflecting data entry errors; values above 100 are excluded from this chart).
- **Attendance rate**: the share of enrolled students present on the survey day. Lower attendance compresses the effective instructional time children receive, even holding class quality constant.
- **Teacher attendance rate**: the share of contracted teachers present on the survey day. Teacher absenteeism is a well-documented driver of learning loss in Pakistan's public schools — a classroom with no teacher delivers no instruction regardless of its other resources.

In [ ]:
METRICS = [
    ('ptr_clean',              'Pupil–Teacher Ratio',       'Mean PTR (lower = better)',       False),
    ('attendance_rate',        'Student Attendance Rate',   'Mean attendance rate (%)',         True),
    ('teacher_attendance_rate','Teacher Attendance Rate',   'Mean teacher attendance rate (%)', True),
]

means = (
    df.groupby('school_type')[['ptr_clean', 'attendance_rate', 'teacher_attendance_rate']]
    .mean()
    .loc[['Government', 'Private']]
)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for ax, (col, title, ylabel, is_pct) in zip(axes, METRICS):
    vals  = [means.loc['Government', col], means.loc['Private', col]]
    colors = [GOVT_COLOR, PVT_COLOR]
    bars = ax.bar(['Government', 'Private'], vals, color=colors,
                  width=0.45, zorder=3, edgecolor='white')

    for bar, v in zip(bars, vals):
        label = f'{v:.1f}%' if is_pct else f'{v:.1f}'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (1.0 if is_pct else 0.3),
            label, ha='center', va='bottom', fontsize=10.5, fontweight='bold'
        )

    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel, labelpad=6)
    ymax = max(vals) * 1.20
    ax.set_ylim(0, ymax)
    if is_pct:
        ax.set_ylim(0, 105)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax.grid(axis='x', visible=False)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

# PTR: add footnote about cap
axes[0].annotate(
    f'Values > {PTR_CAP} excluded (n = {n_extreme})',
    xy=(0.5, -0.14), xycoords='axes fraction',
    ha='center', fontsize=8, color='#666666'
)

fig.suptitle('School Operational Metrics — Government vs Private',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGURES / '06a_operational_metrics.png', bbox_inches='tight')
fig.savefig(FIGURES / '06a_operational_metrics.pdf', bbox_inches='tight')
plt.show()

print(means[['ptr_clean','attendance_rate','teacher_attendance_rate']].round(1))

## Figure 2 — Classroom Resource Availability: Government vs Private

Beyond staffing, the physical learning environment matters. The three classroom-level resources below were observed in Grade 2 classrooms during the school survey. Grade 2 is the critical foundational year for literacy and numeracy: children who lack textbooks or a visible board in this year face a compounded disadvantage that accumulates across all subsequent grades.

The most striking gap is in **textbook availability**: only 18% of government Grade 2 classrooms had textbooks present on the day of observation, compared to 35% in private schools. This does not necessarily mean textbooks were never delivered — they may have been locked away, taken home, or not yet distributed — but a textbook not in the classroom is a textbook not being used.

In [ ]:
RES_COLS = [
    ('class2_has_textbooks',              'Textbooks\npresent'),
    ('class2_has_supplementary_material', 'Supplementary\nmaterial'),
    ('class2_has_blackboard',             'Blackboard\npresent'),
]

res_means = (
    df.groupby('school_type')[[c for c, _ in RES_COLS]]
    .mean()
    .mul(100)
    .loc[['Government', 'Private']]
)

x      = np.arange(len(RES_COLS))
w      = 0.36
labels = [lbl for _, lbl in RES_COLS]
cols   = [c   for c, _   in RES_COLS]

fig, ax = plt.subplots(figsize=(9, 5.5))

bars_g = ax.bar(x - w / 2, res_means.loc['Government', cols],
                width=w, color=GOVT_COLOR, label='Government', zorder=3)
bars_p = ax.bar(x + w / 2, res_means.loc['Private', cols],
                width=w, color=PVT_COLOR,  label='Private',    zorder=3)

for bars in (bars_g, bars_p):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('% of Grade 2 classrooms', labelpad=8)
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_title('Grade 2 Classroom Resources — Government vs Private Schools',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='x', visible=False)
ax.set_axisbelow(True)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '06b_classroom_resources.png', bbox_inches='tight')
fig.savefig(FIGURES / '06b_classroom_resources.pdf', bbox_inches='tight')
plt.show()

print(res_means[cols].round(1))

## Figure 3 — Pupil–Teacher Ratio by Province and School Type

PTR varies substantially across provinces, reflecting differences in teacher recruitment policies, rural school sizes, and demographic pressures. Box plots show the full within-province distribution (excluding PTR > 100), making visible not just the typical school in each province but the spread and presence of high-PTR outliers that the mean conceals.

In every province, the government school distribution sits higher than the private school distribution — but the *degree* of disadvantage varies. Sindh and Punjab stand out for the heaviest right tails among government schools, consistent with those provinces hosting Pakistan's most densely populated urban areas alongside vast underfunded rural school networks.

In [ ]:
ptr_df = df[df['ptr_clean'].notna()].copy()

# Order provinces by median government PTR (descending) for visual clarity
prov_order = (
    ptr_df[ptr_df['school_type'] == 'Government']
    .groupby('province')['ptr_clean']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(13, 6))

sns.boxplot(
    data=ptr_df,
    x='province', y='ptr_clean', hue='school_type',
    order=prov_order,
    hue_order=['Government', 'Private'],
    palette={'Government': GOVT_COLOR, 'Private': PVT_COLOR},
    width=0.55, linewidth=0.9, fliersize=2.5,
    flierprops=dict(alpha=0.35, markeredgewidth=0),
    ax=ax,
)

ax.axhline(40, color='#555555', linewidth=0.8, linestyle='--', zorder=5,
           alpha=0.7, label='PTR = 40 (commonly-cited threshold)')
ax.set_xlabel('Province  (ordered by median government PTR)', labelpad=8)
ax.set_ylabel(f'Pupil–Teacher Ratio  (capped at {PTR_CAP})', labelpad=8)
ax.set_title('Pupil–Teacher Ratio Distribution by Province and School Type',
             fontsize=13, fontweight='bold', pad=12)

handles, labels_leg = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels_leg, frameon=False, fontsize=10,
          loc='upper right')
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '06c_ptr_boxplot_province.png', bbox_inches='tight')
fig.savefig(FIGURES / '06c_ptr_boxplot_province.pdf', bbox_inches='tight')
plt.show()

print('Median PTR by province and school type:')
print(ptr_df.groupby(['province','school_type'])['ptr_clean']
      .median().round(1).unstack().loc[prov_order])
print()
print('ptr_df head:')
print(ptr_df[['province','school_type','ptr_clean']].head())
print('ptr_df tail:')
print(ptr_df[['province','school_type','ptr_clean']].tail())

## Figure 4 — Attendance Rate vs Pupil–Teacher Ratio

A common assumption is that overcrowded schools generate lower student attendance — large class sizes create chaotic learning environments that discourage attendance, or signal to parents that schooling is low quality. The scatter below tests this directly.

The data does **not** support the overcrowding–absenteeism link in Pakistani government schools (r ≈ 0.00, p = 0.82). This finding is substantively important: it means attendance problems in government schools are not primarily a symptom of class size. Other mechanisms — teacher absenteeism (documented in Figure 1), distance to school, household demand for child labour, and school quality signals — are likely the dominant drivers of government school attendance variation.

In [ ]:
scat_df = df[df['ptr_clean'].notna() & df['attendance_rate'].notna()].copy()

fig, ax = plt.subplots(figsize=(9, 6))

for stype, color in [('Government', GOVT_COLOR), ('Private', PVT_COLOR)]:
    sub = scat_df[scat_df['school_type'] == stype]
    ax.scatter(
        sub['ptr_clean'], sub['attendance_rate'],
        color=color, s=12, alpha=0.30, linewidths=0, label=stype,
    )
    # OLS line
    slope, intercept, r_val, p_val, _ = stats.linregress(
        sub['ptr_clean'], sub['attendance_rate']
    )
    x_fit = np.linspace(sub['ptr_clean'].min(), sub['ptr_clean'].max(), 100)
    ax.plot(x_fit, slope * x_fit + intercept,
            color=color, linewidth=2.0, zorder=5)
    # Annotation
    p_str = 'p < 0.001' if p_val < 0.001 else f'p = {p_val:.3f}'
    ax.annotate(
        f'{stype}\nr = {r_val:.3f},  {p_str}\nn = {len(sub):,}',
        xy=(sub['ptr_clean'].quantile(0.92), slope * sub['ptr_clean'].quantile(0.92) + intercept),
        xytext=(72 if stype == 'Government' else 50,
                62  if stype == 'Government' else 78),
        fontsize=9, color=color,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  alpha=0.85, edgecolor=color, linewidth=0.8),
        arrowprops=dict(arrowstyle='->', color=color, lw=0.9),
    )

ax.set_xlabel(f'Pupil–Teacher Ratio  (capped at {PTR_CAP})', labelpad=8)
ax.set_ylabel('Student Attendance Rate (%)', labelpad=8)
ax.set_title('Attendance Rate vs Pupil–Teacher Ratio by School Type',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(frameon=False, fontsize=10, markerscale=2.5)
sns.despine(ax=ax)
plt.tight_layout()

fig.savefig(FIGURES / '06d_attendance_vs_ptr.png', bbox_inches='tight')
fig.savefig(FIGURES / '06d_attendance_vs_ptr.pdf', bbox_inches='tight')
plt.show()

In [ ]:
govt = df[df['school_type'] == 'Government']['ptr_clean'].dropna()
print(f"National Govt PTR — mean:   {govt.mean():.2f}")
print(f"National Govt PTR — median: {govt.median():.2f}")
print(f"National Govt PTR — n schools: {len(govt)}")